# Project __ Medical-RAG-Chatbot 

## Phase 1 — Experimentation (Jupyter Notebook)

### 1. Objective
- The objective of Phase 1 is to experimentally develop and validate the core components of the **Medical-RAG-Chatbot** before moving to a production-oriented GenAI pipeline.

- In this phase, the complete RAG workflow is implemented and tested inside a Jupyter Notebook.

    - Each component is evaluated independently to ensure that:

        * medical documents can be loaded;
        * text can be appropriately split;
        * embeddings can be generated;
        * the vector database can be created;
        * relevant medical information can be retrieved;
        * the LLM can generate responses using retrieved context;
        * unsupported questions can be handled appropriately.

### 1. Load the Groq API KEY From .env file

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

# Read a Groq and Pinecone API key from .env file
GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
PINECONE_API_KEY=os.environ.get('PINECONE_API_KEY')

In [3]:
# save Groq and pinecone Api Key into our environment 
os.environ["GROQ_API_KEY"] = GROQ_API_KEY
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY


### 2. Load Medical PDF Document

In [4]:
# Check where our currently are before/after
import os
print(os.getcwd())

e:\Data Science\ALL Projects\GenAI Projects\Medical-RAG-Chatbot\Notebook\01 Phase 1 Notebook Experiment


In [5]:

""" 
Medical-RAG-Chatbot/
│
├── data/
│   └── Medical_book.pdf
│
└── Notebook/
    └── 01 Phase 1 Notebook Experiment/
        └──  This is the current position 
        
    - (so we need  step back, to get the root directory, which is Medical-RAG-Chatbot)

""" 



' \nMedical-RAG-Chatbot/\n│\n├── data/\n│   └── Medical_book.pdf\n│\n└── Notebook/\n    └── 01 Phase 1 Notebook Experiment/\n        └──  This is the current position \n\n    - (so we need  step back, to get the root directory, which is Medical-RAG-Chatbot)\n\n'

In [6]:

# import os
# if os.path.basename(os.getcwd()) != "Medical-RAG-Chatbot":
#     os.chdir("../")

In [12]:
# Change Directory 

import os

target = "Medical-RAG-Chatbot"

while True:
    current = os.path.basename(os.getcwd())

    if current == target:
        print("Now in:", os.getcwd())
        break  # <-- stops the loop once found

    parent = os.path.dirname(os.getcwd())

    # If going up doesn't change anything, we've hit the drive root — stop
    if parent == os.getcwd():
        print(f"Reached root without finding '{target}'. Stopping.")
        break

    os.chdir("../")

# another syntax: pdf_file_path = "../data/Medical_book.pdf" 

Now in: e:\Data Science\ALL Projects\GenAI Projects\Medical-RAG-Chatbot


In [13]:
%pwd

'e:\\Data Science\\ALL Projects\\GenAI Projects\\Medical-RAG-Chatbot'

In [ ]:
# import Library
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader

# Extract Data
def load_pdf_file(data):
    """
    Function : load_pdf_file
        - purpose: Split the all pdf data into single document
        
        - input: Directory of pdf file path
        
        - output: All Documents into single docs file
    """
    loader = DirectoryLoader(
        data,
        glob = "*.pdf",
        loader_cls = PyPDFLoader
    )
    documents = loader.load()

    return documents

In [15]:
extracted_data = load_pdf_file(data="data/")

print(len(extracted_data))

637


In [16]:
# first three values
extracted_data[:3]

[Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 0}, page_content=''),
 Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 1}, page_content='TheGALE\nENCYCLOPEDIA\nofMEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'data\\Medical_book.pdf', 'page': 2}, page_content='TheGALE\nENCYCLOPEDIA\nofMEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B1')]

#### Organize the All Document__ only count the page_content and source in our Document

In [ ]:
# only extract the page_content and source

from langchain_core.documents import Document
from typing import List

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Function : filter_to_minimal_docs
        - purpose: Filter the document (only extract the page_content and source)
        
        - input: list of Document that include some metadata
        
        - output: Filtered of list of Document that have only page_content and source
    """
    minimal_docs: List[Document] = []

    # where, docs : list of old document that have some extra information
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content = doc.page_content,
                metadata = {"source":src}

            )
        )

    return minimal_docs


In [18]:
minimal_docs = filter_to_minimal_docs(extracted_data)

print(f"Length of document: {len(minimal_docs)} pages")

Length of document: 637 pages


In [19]:
# first five values
minimal_docs[:5]

[Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content=''),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='TheGALE\nENCYCLOPEDIA\nofMEDICINE\nSECOND EDITION'),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='TheGALE\nENCYCLOPEDIA\nofMEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B1'),
 Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow, Manager, Imaging and Multimedia\nContent\nRobyn V . Young, Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imaging and\nMultimedia C

### 3. Chunk Distribution

In [20]:
# split the data into Text chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

def text_split(minimal_docs):
    """
    Function : text_split
        - purpose: split the minimal docs document into smaller chunk 
        
        - input: list of Document
        
        - return: list of chunk document
    """
    text_splitter = RecursiveCharacterTextSplitter(chunk_size = 600, chunk_overlap = 25)

    text_chunks = text_splitter.split_documents(minimal_docs)

    return text_chunks

In [21]:
text_chunks = text_split(minimal_docs)

print(f"Length of Text Chunks: {len(text_chunks)}")

print(f"First five chunK: \n{text_chunks[:4]}")


Length of Text Chunks: 5657
First five chunK: 
[Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='TheGALE\nENCYCLOPEDIA\nofMEDICINE\nSECOND EDITION'), Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='TheGALE\nENCYCLOPEDIA\nofMEDICINE\nSECOND EDITION\nJACQUELINE L. LONGE, EDITOR\nDEIRDRE S. BLANCHFIELD, ASSOCIATE EDITOR\nVOLUME\nA-B1'), Document(metadata={'source': 'data\\Medical_book.pdf'}, page_content='STAFF\nJacqueline L. Longe, Project Editor\nDeirdre S. Blanchfield, Associate Editor\nChristine B. Jeryan, Managing Editor\nDonna Olendorf, Senior Editor\nStacey Blachford, Associate Editor\nKate Kretschmann, Melissa C. McDade, Ryan\nThomason, Assistant Editors\nMark Springer, Technical Specialist\nAndrea Lopeman, Programmer/Analyst\nBarbara J. Yarrow, Manager, Imaging and Multimedia\nContent\nRobyn V . Young, Project Manager, Imaging and\nMultimedia Content\nDean Dauphinais, Senior Editor, Imaging and\nMultimedia Content\nKelly A. Quin, Editor,

### 4. Download the Embedding model form the Hugging Face

- model_name="sentence-transformers/all-MiniLM-L6-v2"

In [22]:
from langchain_huggingface import HuggingFaceEmbeddings

def download_hugging_face_embeddings():
    embeddings = HuggingFaceEmbeddings(
        model_name="sentence-transformers/all-MiniLM-L6-v2"
    )

    return embeddings

# create an an object of embedding model
embeddings = download_hugging_face_embeddings()

c:\Users\kz\anaconda3\envs\llmapp\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\kz\anaconda3\envs\llmapp\Lib\site-packages\langchain_huggingface\chat_models\__init__.py:1: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain_core.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain_core.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  from langchain_huggingface.chat_models.huggingface import (


In [23]:
# Test the hugging face model
query_result = embeddings.embed_query("Kz Raihan")

print("Length: ", len(query_result))

Length:  384


In [24]:
query_result[:5]

[-0.037495069205760956,
 0.057029418647289276,
 -0.06345462054014206,
 -0.014790771529078484,
 -0.025175541639328003]

- Key Note:
    - Hugging Face "sentence-transformers/all-MiniLM-L6-v2" embedding model each world dimension: 384

### 5. Create a Pinecone Vector Database

In [25]:
from pinecone import Pinecone

# 1. Load API key from environment
api_key = os.getenv("PINECONE_API_KEY")
if not api_key:
    raise ValueError("PINECONE_API_KEY not set! Run: export PINECONE_API_KEY='...'")

# 2. Initialize modern SDK client    
pc = Pinecone(api_key=api_key)

# 3. List existing indexes (proves connection works)
indexes = pc.list_indexes()
print("Connected! Your Pinecone indexes:")
print(indexes)


Connected! Your Pinecone indexes:
[{
    "name": "langchain-pinecone",
    "metric": "cosine",
    "host": "langchain-pinecone-5nhvocp.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}, {
    "name": "medical-bot",
    "metric": "cosine",
    "host": "medical-bot-5nhvocp.svc.aped-4627-b74a.pinecone.io",
    "spec": {
        "serverless": {
            "cloud": "aws",
            "region": "us-east-1"
        }
    },
    "status": {
        "ready": true,
        "state": "Ready"
    },
    "vector_type": "dense",
    "dimension": 384,
    "deletion_protection": "disabled",
    "tags": null
}]


#### Step 1: Create an index 

In [26]:
from pinecone import ServerlessSpec

index_name = "medical-bot"

if not pc.has_index(index_name):
    pc.create_index(
        name = index_name,
        dimension = 384,
        metric = "cosine",
        spec = ServerlessSpec(cloud = "aws", region = "us-east-1")
    )

# Connect to Pinecone Index
index = pc.Index(index_name)

#### Step 2: Store Data into Pinecone Vector DataBase

In [28]:
# Embed each chunk and upsert the embeddings into our Pinecone index
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    index_name = index_name,
    embedding=embeddings
)


#### Step 3: Load the Existing index

In [ ]:
from langchain_pinecone import PineconeVectorStore

# Load each chunk to our local Machine 
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)


In [ ]:
# Access the underlying Pinecone index object
index = docsearch._index  

stats = index.describe_index_stats()
print(stats)

{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 5657}},
 'total_vector_count': 5657,
 'vector_type': 'dense'}


In [35]:
stats = index.describe_index_stats()
print(f"Total vectors stored: {stats['total_vector_count']}")

Total vectors stored: 5657


#### Step 4: Add more data to existing pinecone database

In [39]:
# create the data
add_data = Document(
    page_content="My name is KzRaihan",
    metadata = {"source": "Linkedlin"}
)


In [40]:
# store the data 
docsearch.add_documents(documents=[add_data])

['bc317708-bf2b-4f4f-b08c-869c3fae65ef']

### 6. Create an vector store retriever

In [41]:
# Create Retriever
retriever = docsearch.as_retriever(
    search_type = "similarity", # By Default
    search_kwargs = {
        "k": 3,
    }

)

In [45]:
retriever_docs = retriever.invoke("What is Acne?")

retriever_docs

[Document(id='9b4ef84d-f91e-4221-82c1-99d440c26cca', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when thepores of the skin become clogged with oil, dead skincells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin disease. It affects nearly 17 millionpeople in the United States. While acne can arise at anyage, it usually begins at puberty and worsens during ado-\nlescence. Nearly 85% of people develop acne at some timebetween the ages of 12-25 years. Up to 20% of womendevelop mild acne. It is also found in some newborns.'),
 Document(id='39fc0cc3-0dbb-48c7-af38-1d680c9b0fa3', metadata={'source': 'data\\Medical_book.pdf'}, page_content='McPhee, et al. Stamford: Appleton & Lange, 1995.\nKaptchuk, Ted J., Z’ev Rosenberg, and K’an Herb Co., Inc.\nK’an Herbals: Formulas by Ted Kaptchuk, O.M.D. San\nFrancisco: A

### 7. Generate Response

#### Step 1: Initialize the LLM

In [51]:
from langchain_groq import ChatGroq

In [52]:
chatModel = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.3
)


#### Step 2: Define the Prompt

In [53]:
from langchain_core.prompts import ChatPromptTemplate

In [55]:
system_prompt = (
    "You are an assistant for question-answering tasks. "
    "Use the following pieces of retrieved context to answer "
    "the question. If you don't know the answer, say that you "
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system_prompt),
        ("human", "{input}")
    ]
)

#### Step 3: Create Chain

In [57]:
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser


# ------------------------------------------------------------
# Helper to combine retrieved documents into a single context string
# ------------------------------------------------------------

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# ------------------------------------------------------------
# Equivalent to: create_stuff_documents_chain(chatModel, prompt)
#
# Takes {"context": [Document, ...], "input": "..."} and returns
# the LLM's generated answer as a plain string.
# ------------------------------------------------------------

question_answer_chain = (
    {
        "context": lambda x: format_docs(x["context"]),
        "input": lambda x: x["input"],
    }
    | prompt
    | chatModel
    | StrOutputParser()
)

# ------------------------------------------------------------
# Equivalent to: create_retrieval_chain(retriever, question_answer_chain)
#
# Step 1: retrieve documents for the input query, add them under "context"
# Step 2: pass the full dict (input + context) into question_answer_chain,
#         add the result under "answer"
# ------------------------------------------------------------

rag_chain = (
    RunnablePassthrough.assign(
        context=RunnableLambda(lambda x: retriever.invoke(x["input"]))
    )
    | RunnablePassthrough.assign(
        answer=question_answer_chain
    )
)

In [58]:
response = rag_chain.invoke({"input": "What is diabetic retinopathy?"})

print(response["answer"])    # the generated answer (string)
print(response["context"])   # the raw retrieved Document objects

Diabetic retinopathy is a complication of diabetes that damages the blood vessels in the retina. High blood sugar levels cause these vessels to leak, bleed, or become blocked, leading to vision loss. It is one of the leading causes of blindness in adults and requires early detection and control of blood glucose.
[Document(id='ffde167c-0353-4535-b32d-4567c87f86a8', metadata={'source': 'data\\Medical_book.pdf'}, page_content='line with the center of the cornea. Diabetes can play a rolein the development of astigmatism. High blood sugar lev-els can cause shape changes in the lens of the eye. Thisprocess usually occurs slowly and, often, is only noticedwhen the diabetic has started treatment to control theirblood sugar. The return to a more normal blood sugarallows the lens to return to normal and this change issometimes noticed by the patient as farsightedness.Because of this, diabetics should wait until their bloodsugar is under control for at least one month to allowvision to stabilize 